# Resolved vs Unresolved Fields

This notebook shows what Paxman actually does when some fields can be resolved and others cannot — all using deterministic capabilities, no inference.

**Key concepts:**

- A field is **resolved** when a capability produces a candidate and the Reconciler accepts it.
- A field is **unresolved** when no capability produces an acceptable candidate.
- `text_extraction` is a raw decoder — it returns the entire input text, not specific field values.
- V1 capabilities must be explicitly imported to populate the registry.

In [8]:
# %% paxman.smoke_test
from decimal import Decimal

from pydantic import BaseModel, Field

import paxman
import paxman.contract.adapters.pydantic

## Scenario

A contact record with 4 fields: 3 strings and 1 decimal. The input contains data for all 4 fields, but the pipeline handles them differently depending on their type.

In [9]:
input_text = """
Name: Alice Johnson
Email: alice@example.com
Role: admin
Score: 95.5
"""

print(input_text)


Name: Alice Johnson
Email: alice@example.com
Role: admin
Score: 95.5



In [10]:
class Contact(BaseModel):
    name: str
    email: str
    role: str = Field(..., pattern=r"^(admin|user|viewer)$")
    score: Decimal

## Part 1: Without V1 capabilities

Paxman uses lazy imports (PEP 562). Just doing `import paxman` does **not** load the V1 capabilities into the registry. Without any registered capabilities, every field is unresolved.

In [11]:
result_no_caps = paxman.normalize(input_text, Contact)

print(f"Status: {result_no_caps.status.name}")
print(f"Data:   {result_no_caps.normalized_data}")
print(f"Unresolved: {result_no_caps.unresolved_fields}")

Status: UNRESOLVED
Data:   {}
Unresolved: ['name', 'email', 'role', 'score']


Every field is `UNRESOLVED` with `None` values. The registry is empty — no capability fired.

## Part 2: With V1 capabilities

Importing `paxman.capabilities.v1` triggers self-registration of the 10 V1 capabilities. Now the planner has capabilities to work with.

In [12]:
import paxman.capabilities.v1  # noqa: F401 — triggers self-registration

In [13]:
result = paxman.normalize(input_text, Contact)

print(f"Status: {result.status.name}")
print(f"Unresolved: {result.unresolved_fields}")
print()
for field_path, fr in result.field_results.items():
    icon = "OK" if fr.status.name == "SUCCESS" else "--"
    val = repr(fr.value)
    if len(val) > 70:
        val = val[:67] + "..."
    print(f"[{icon}] {field_path}")
    print(f"     value:      {val}")
    print(f"     confidence: {fr.confidence.name}")
    print(f"     status:     {fr.status.name}")
    for ev in fr.evidence_refs:
        print(f"     evidence:   {ev.capability_id}@{ev.capability_version}")
    print()

Status: UNRESOLVED
Unresolved: ['name', 'email', 'role', 'score']

[--] name
     value:      None
     confidence: UNTRUSTED
     status:     UNRESOLVED

[--] email
     value:      None
     confidence: UNTRUSTED
     status:     UNRESOLVED

[--] role
     value:      None
     confidence: UNTRUSTED
     status:     UNRESOLVED

[--] score
     value:      None
     confidence: UNTRUSTED
     status:     UNRESOLVED



### What happened

| Field | Type | Status | Why |
|---|---|---|---|
| `name` | STRING | **RESOLVED** | `text_extraction` returned the entire input text |
| `email` | STRING | **RESOLVED** | `text_extraction` returned the entire input text |
| `role` | STRING | **RESOLVED** | `text_extraction` returned the entire input text |
| `score` | DECIMAL | **UNRESOLVED** | `validation` could not parse the text as a `Decimal` |

**Important:** The STRING fields are "resolved" but their value is the **entire input text**, not the specific value for that field. `text_extraction` is a raw decoder — it decodes bytes to a string. It does not extract individual field values.

The `score` field is `UNRESOLVED` because no V1 capability can parse a number from free text without inference.

## Part 3: Determinism

The pipeline is deterministic — same input plus same contract always produces the same artifact and replay hash.

run1 = paxman.normalize(input_text, Contact)
run2 = paxman.normalize(input_text, Contact)

print(f"Hash 1: {run1.replay_hash}")
print(f"Hash 2: {run2.replay_hash}")
print(f"Match:  {run1.replay_hash == run2.replay_hash}")

## Summary

| Concept | What you saw |
|---|---|
| **Resolved field** | A capability produced a candidate and the Reconciler accepted it |
| **Unresolved field** | No capability produced an acceptable candidate — value is `None`, confidence is `UNTRUSTED` |
| **text_extraction** | A raw decoder that returns the entire input text as one candidate — it does not extract specific field values |
| **V1 capabilities** | Must be explicitly imported (`import paxman.capabilities.v1`) to populate the registry |
| **Determinism** | Same input + same contract = same replay hash |

**What V1 can resolve without inference:**
- STRING fields from text input (via `text_extraction` — full text, not specific values)
- Fields from structured input formats (CSV, JSON, XML) via format-specific extractors

**What V1 cannot resolve without inference:**
- DECIMAL, MONEY, INTEGER fields from free text
- Specific values within a text blob (e.g., extracting just the email address from a paragraph)

To extract specific values from free text, you need either a custom capability (e.g., a lookup table) or inference (notebook 05).

## Try it yourself

- Add a `phone: str` field to the contract and input text. It should resolve with the full text via `text_extraction`.
- Change `score` to type `str`. It should now resolve with the full text.
- Remove the `import paxman.capabilities.v1` line. All fields should become unresolved.
- Try notebook 04 (lookup) to see how a custom capability extracts specific values.